# 02 - Spectral Indices

**Goal**: Visualise all 7 spectral indices on the 2023 Lanzarote composite and confirm
each one highlights the expected land cover type.

The index code lives in `pipeline/indices.py` - this notebook is the visual proof that it works.

| Index | What it measures | Key signal on Lanzarote |
|---|---|---|
| NDVI  | Vegetation density | La Geria vineyards, Famara scrub |
| NDWI  | Open water | Salinas (salt flats), Atlantic coast |
| NDBI  | Built-up / impervious | Arrecife, Puerto del Carmen resort strip |
| SAVI  | Sparse vegetation (soil-adjusted) | Better than NDVI for arid volcanic terrain |
| BSI   | Bare soil / rock | Timanfaya malpais, Fuerteventura dunes |
| EVI   | Dense vegetation (atmospheric corrected) | Complements NDVI in high-biomass patches |
| MNDWI | Water vs. urban (modified) | Distinguishes salinas from coastal development |

**Phase**: 1 - Data exploration  
**Data**: Landsat 9 OLI-2, dry-season composite May–Sep 2023


In [ ]:
import sys
import os

project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import ee
import folium
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from IPython.display import display

from pipeline.indices import gee_add_indices

print('Libraries loaded OK')


In [ ]:
GEE_PROJECT = 'project-4cb2ec4e-f113-48f1-8b5'
AOI_COORDS  = [-13.92, 28.80, -13.30, 29.30]
MAP_CENTRE  = [29.05, -13.61]
MAP_ZOOM    = 10

ee.Initialize(project=GEE_PROJECT)
aoi = ee.Geometry.Rectangle(AOI_COORDS)
print('GEE initialised')


In [ ]:
# ── Load 2023 dry-season composite ────────────────────────────────────────────
def mask_clouds(image):
    qa = image.select('QA_PIXEL')
    return image.updateMask(
        qa.bitwiseAnd(1 << 3).eq(0).And(qa.bitwiseAnd(1 << 4).eq(0))
    )

def scale_sr(image):
    optical = image.select('SR_B.').multiply(0.0000275).add(-0.2)
    return image.addBands(optical, overwrite=True)

composite_2023 = (
    ee.ImageCollection('LANDSAT/LC09/C02/T1_L2')
    .filterBounds(aoi)
    .filterDate('2023-05-01', '2023-09-30')
    .filter(ee.Filter.lt('CLOUD_COVER', 20))
    .map(mask_clouds)
    .map(scale_sr)
    .select(['SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B6', 'SR_B7'],
            ['blue',  'green', 'red',   'nir',   'swir1', 'swir2'])
    .median()
    .clip(aoi)
)

# Add all 7 indices using pipeline/indices.py
composite_with_indices = gee_add_indices(composite_2023)

print('Composite + indices ready')
print('Bands available:', composite_with_indices.bandNames().getInfo())


## 1. Interactive Map - All Indices as Toggleable Layers

Use the layer control (top-right) to switch between indices.
Toggle the base satellite imagery on/off for context.


In [ ]:
def add_ee_layer(folium_map, ee_image, vis_params, name, shown=False, opacity=0.85):
    map_id_dict = ee.Image(ee_image).getMapId(vis_params)
    folium.raster_layers.TileLayer(
        tiles=map_id_dict['tile_fetcher'].url_format,
        attr='Map data © Google Earth Engine / USGS',
        name=name, overlay=True, control=True,
        show=shown, opacity=opacity,
    ).add_to(folium_map)

# Shared diverging palette: red (low/negative) → white (zero) → green (high/positive)
div_palette  = ['#d73027', '#f46d43', '#fdae61', '#ffffbf', '#a6d96a', '#66bd63', '#1a9850']
# Water palette: brown → white → blue
water_palette = ['#8c510a', '#f5f5f5', '#01665e', '#2166ac']

index_layers = [
    ('ndvi',  {'min': -0.2, 'max': 0.6,  'palette': div_palette},   'NDVI - Vegetation',        True),
    ('ndwi',  {'min': -0.4, 'max': 0.4,  'palette': water_palette},  'NDWI - Water bodies',      False),
    ('ndbi',  {'min': -0.3, 'max': 0.3,  'palette': div_palette[::-1]}, 'NDBI - Built-up',       False),
    ('savi',  {'min': -0.2, 'max': 0.5,  'palette': div_palette},   'SAVI - Sparse vegetation', False),
    ('bsi',   {'min': -0.3, 'max': 0.3,  'palette': div_palette[::-1]}, 'BSI - Bare soil/rock',  False),
    ('evi',   {'min': -0.1, 'max': 0.5,  'palette': div_palette},   'EVI - Enhanced vegetation',False),
    ('mndwi', {'min': -0.4, 'max': 0.4,  'palette': water_palette},  'MNDWI - Water vs. urban', False),
]

# True-colour base for reference
true_colour_vis = {'bands': ['red', 'green', 'blue'], 'min': 0.0, 'max': 0.3, 'gamma': 1.4}

m = folium.Map(location=MAP_CENTRE, zoom_start=MAP_ZOOM, tiles='CartoDB dark_matter')
add_ee_layer(m, composite_2023, true_colour_vis, 'True colour (reference)', shown=False, opacity=1.0)

for band, vis, label, shown in index_layers:
    add_ee_layer(m, composite_with_indices.select(band), vis, label, shown=shown)

folium.LayerControl(collapsed=False).add_to(m)
display(m)


## 2. Spot-Check: Index Values at Known Locations

Verify that each index returns sensible values where we know ground truth.


In [ ]:
spot_checks = {
    'Arrecife (urban)':         (28.9631, -13.5495),
    'Puerto del Carmen (resort)':(28.9197, -13.6571),
    'Timanfaya (volcanic rock)': (29.0031, -13.7557),
    'La Geria (vineyards)':      (29.0250, -13.6700),
    'Famara (scrubland)':        (29.1800, -13.5700),
    'Salinas de Janubio':        (28.9350, -13.8280),
    'Atlantic Ocean':            (28.8500, -13.7000),
}

indices = ['ndvi', 'ndwi', 'ndbi', 'savi', 'bsi', 'evi', 'mndwi']

print(f"{'Location':<30} {'NDVI':>6} {'NDWI':>6} {'NDBI':>6} {'SAVI':>6} {'BSI':>6} {'EVI':>6} {'MNDWI':>6}")
print('-' * 78)

for location, (lat, lon) in spot_checks.items():
    point = ee.Geometry.Point([lon, lat])
    vals = (
        composite_with_indices
        .select(indices)
        .sample(point, 30)
        .first()
        .toDictionary()
        .getInfo()
    )
    row = '  '.join(f"{vals.get(idx, float('nan')):>5.3f}" for idx in indices)
    print(f"{location:<30}  {row}")


## 3. Static Grid - All 7 Indices Side by Side

Downloads a small sample region (Arrecife + surroundings) for a static matplotlib comparison.
This is slower (~1 min) but produces a saveable figure.


In [ ]:
# Small AOI around Arrecife + coast (faster to download than full island)
sample_aoi = ee.Geometry.Rectangle([-13.62, 28.90, -13.46, 29.02])

index_cfg = [
    ('ndvi',  (-0.2, 0.6),  'RdYlGn',    'NDVI\n(vegetation)'),
    ('ndwi',  (-0.4, 0.4),  'BrBG',      'NDWI\n(water)'),
    ('ndbi',  (-0.3, 0.3),  'RdYlGn_r',  'NDBI\n(built-up)'),
    ('savi',  (-0.2, 0.5),  'RdYlGn',    'SAVI\n(sparse veg)'),
    ('bsi',   (-0.3, 0.3),  'RdYlGn_r',  'BSI\n(bare soil)'),
    ('evi',   (-0.1, 0.5),  'RdYlGn',    'EVI\n(enhanced veg)'),
    ('mndwi', (-0.4, 0.4),  'BrBG',      'MNDWI\n(water vs urban)'),
]

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for ax_idx, (band, (vmin, vmax), cmap, title) in enumerate(index_cfg):
    print(f'Downloading {band}...', end=' ', flush=True)
    arr = composite_with_indices.select(band).sampleRectangle(
        region=sample_aoi, defaultValue=0
    ).get(band).getInfo()
    data = np.array(arr, dtype=float)
    data[data == 0] = np.nan   # mask fill values at edges

    im = axes[ax_idx].imshow(data, cmap=cmap, vmin=vmin, vmax=vmax,
                              interpolation='nearest', aspect='auto')
    axes[ax_idx].set_title(title, fontsize=11, fontweight='bold')
    axes[ax_idx].axis('off')
    fig.colorbar(im, ax=axes[ax_idx], fraction=0.046, pad=0.04)
    print('done')

# Last subplot: true colour reference
print('Downloading true colour...', end=' ', flush=True)
rgb_bands = ['red', 'green', 'blue']
rgb_data = []
for b in rgb_bands:
    arr = composite_2023.select(b).sampleRectangle(
        region=sample_aoi, defaultValue=0
    ).get(b).getInfo()
    rgb_data.append(np.array(arr, dtype=float))

rgb = np.stack(rgb_data, axis=-1)
rgb = np.clip(rgb / 0.3, 0, 1)   # normalise to 0–1 for display
axes[7].imshow(rgb, aspect='auto')
axes[7].set_title('True colour\n(reference)', fontsize=11, fontweight='bold')
axes[7].axis('off')
print('done')

fig.suptitle('Spectral Indices - Arrecife area, Lanzarote (2023)',
             fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('../data/processed/spectral_indices_grid_2023.png',
            dpi=150, bbox_inches='tight')
plt.show()
print('\nSaved to data/processed/spectral_indices_grid_2023.png')


## 4. Next Steps

With all 7 indices confirmed working and validated against known locations, we're ready for **Phase 2**:

**Notebook 03 - Classification**
1. Download CORINE Land Cover labels for the Canary Islands
2. Remap 44 CORINE classes → our 6 simplified classes
3. Build feature vectors: `[blue, green, red, nir, swir1, swir2, ndvi, ndwi, ndbi, savi, bsi, evi, mndwi]`
4. Train Random Forest classifier
5. Evaluate: confusion matrix, Overall Accuracy, Kappa coefficient

---
*Data: Landsat 9 OLI-2 Collection 2 L2, dry-season composite May–Sep 2023*  
*Indices implemented in `pipeline/indices.py`*


In [ ]:
# TODO (Phase 2)
# Import pipeline.indices and demonstrate each index on the 2023 composite
print('Notebook 02 - not yet implemented')